## Heavy Tailed Metrics
Problems with heavy tailed metrics is that few extreme values can increase variance considerably leading to the decrease in power or increase in required sample size. In this exercise we will use winsorize and log-transformation to deal with this problem. Our objective is to identify when to pick the raw, winsorize or log-transformation

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
experiment = pd.read_csv("../data/experiment.csv", parse_dates = ["exposure_date"])
experiment.head()

,user_id,variant,exposure_date,pre_gmv_28d,orders_14d,gmv_14d
0,14357,treatment,2026-09-04,1534.52,4,1441.69
1,3440,control,2026-09-04,1817.86,1,368.36
2,12154,control,2026-09-01,0.00,1,421.78
3,15030,treatment,2026-09-06,0.00,1,258.44
4,18550,treatment,2026-09-01,518.17,1,166.94


### 1. Data - Control Arm Profile

In [3]:
control_df = experiment[experiment["variant"] == "control"].copy()

# Population size
pop = control_df["gmv_14d"].to_numpy()
total_users = len(pop)
print(f"Population size: {len(pop)}")

# Zero GMV Users share
no_order_users = len(control_df[control_df["gmv_14d"] == 0])
share_of_zeroes = no_order_users/total_users
print(f"Share of Zeroes: {share_of_zeroes*100:.2f}%")

# Median, p90, p99 and max GMV
p50, p90, p99, p100 = np.quantile(control_df["gmv_14d"], [0.5, 0.9, 0.99, 1])
print(f"Median: {p50:.4f}, 90th Percentile: {p90:.4f}, 99th Percentile: {p99:.4f}, Max: {p100:.4f}")

# Squared Deviation contribution by top 1%
control_df["is_top_1pct"] = control_df["gmv_14d"] > p99
mean = control_df["gmv_14d"].mean()
control_df["squared_dev_from_mean"] = (control_df["gmv_14d"] - mean)**2

top_std_share = control_df[control_df["is_top_1pct"]]["squared_dev_from_mean"].sum()/control_df["squared_dev_from_mean"].sum()
print(f"Share of Top 1% Squared Deviation: {top_std_share*100:.2f}%")

Population size: 5702
Share of Zeroes: 40.32%
Median: 192.6500, 90th Percentile: 967.0550, 99th Percentile: 2562.3960, Max: 9103.1400
Share of Top 1% Squared Deviation: 38.76%


### 2. Simulator

In [4]:
def simulate_power(pop, n, scenario, sims, seed):
    """
    Share of simulated experiments where each metric gives p < 0.05
    """
    rng = np.random.default_rng(seed)
    rejections = {"raw": 0, "winsorized": 0, "log1p": 0}

    p95 = np.quantile(pop, 0.95) 
    top_share = (pop >= p95).mean()
    tail_add = 0.05 * pop.mean() / top_share

    for _ in range(sims):
        # (a) one simulated experiment: two independent arms from the same population
        c = rng.choice(pop, size=n, replace=True)
        t = rng.choice(pop, size=n, replace=True)

        # (b) inject the truth: only the treatment arm changes
        if scenario == "aa":
            pass                                       # no effect
        elif scenario == "mult":
            t = t * 1.05                               # everyone +5%
        elif scenario == "tail":
            t = np.where(t >= p95, t + tail_add, t)    # only big spenders gain
        else:
            raise ValueError(f"unknown scenario: {scenario}")

        # (c) pooled cap: p99 of both arms together, same cap for both
        cap = np.quantile(np.concatenate([c, t]), 0.99)

        # (d) three versions of the metric for each arm
        versions = {
            "raw":        (c, t),
            "winsorized": (np.minimum(c, cap), np.minimum(t, cap)),
            "log1p":      (np.log1p(c), np.log1p(t)),
        }

        # (e) one Welch test per version; count rejections
        for metric, (cc, tt) in versions.items():
            p = stats.ttest_ind(tt, cc, equal_var=False).pvalue
            if p < 0.05:
                rejections[metric] += 1

    return {metric: count / sims for metric, count in rejections.items()}

### 3. Method
Resample two arms of 5000 users with 1000 simulations for three scenarios
1. A/A Testing: no effect.
2. Multiplicative: every treatment user’s GMV × 1.05.
3. Tail-only: add a constant to treatment users at or above the population’s p95. Derive the constant so the mean rises by the same 5%.

In [5]:
results = []
simulations = 1000

for s in ["aa", "mult", "tail"]:
    result = simulate_power(pop, n=5000, scenario=s, sims=simulations, seed=0)
    result["scenario"] = s
    results.append(result)

result_df = pd.DataFrame(results)
result_df = result_df.set_index(["scenario"])
result_df

,raw,winsorized,log1p
scenario,,,
aa,0.041,0.046,0.057
mult,0.322,0.342,0.077
tail,0.306,0.268,0.059


### 4. Monte Carlo CI on the A/A row

In [6]:
f = result_df.loc["aa"]
se = np.sqrt(f * (1 - f) / simulations)
print(pd.DataFrame({"fpr": f, "lo": f - 1.96 * se, "hi": f + 1.96 * se}).round(3))

              fpr     lo     hi
raw         0.041  0.029  0.053
winsorized  0.046  0.033  0.059
log1p       0.057  0.043  0.071


Every metric's Confidence Interval contains 0.05 which means every metric holds its α in A/A test.

### 5. Variance: winsorized vs raw

In [7]:
cap = np.quantile(pop, 0.99)
print("winsorized variance / raw variance:",
      round(np.minimum(pop, cap).var(ddof=1) / pop.var(ddof=1), 3))

winsorized variance / raw variance: 0.77


### 6. Conclusion
1. for Multiplicative scenario has the most power when everyone gets +5%, winsorized (0.342) has slightly higher power than raw (0.322). Because  winsorized caps top share to c, reducing some of their lift and the noise drops a lot by capping of extreme values, by about 23%.
2. for Tail-only scenario where only big spenders gets constant added to their values, raw (0.306) has higher power than winsorized (0.268). Because winsorized caps top share to c, and in this scenario that’s exactly where the lift was added
3. log1p is nearly powerless here because 40% zeros stay at 0 while buyers sit around 6 on the log scale; that gap dominates the variance and swamps a 5% lift.